## 통계 회귀 모델 실습(3)

1. 가설 설정

2. 데이터 수집 및 전처리

* (1) 데이터 수집

In [ ]:
!git clone 'https://'github.com/KangKyungHwa/DSMH_2/''

# 1. 나눔고딕 폰트 설치
!sudo apt-get update -qq
!sudo apt-get install -y fonts-nanum-extra -qq

# 2. 시스템 폰트 캐시 업데이트
!sudo fc-cache -fv

# 3. Matplotlib 폰트 캐시 삭제
!rm -rf ~/.cache/matplotlib

from matplotlib import pyplot as plt

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (6, 6)
plt.rcParams['axes.unicode_minus'] = False

Cloning into 'DSMH_2'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 83 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 1.36 MiB | 6.26 MiB/s, done.
Resolving deltas: 100% (21/21), done.
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 2.)
debconf: falling back to frontend: Readline
debco

In [ ]:
import pandas as pd # 데이터 로드 및 전처리에 사용
import statsmodels.api as sm # 통계적 회귀 모델 생성 및 분석에 사용
from sklearn.model_selection import train_test_split # 데이터셋을 훈련용과 테스트용으로 분할

data = pd.read_csv("DSMH_2/9_10/housing.csv")

print(data.head(3), '\n\n')
print(data.tail(3))

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY   


       longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
20637    -121.22     39.43                17.0       2254.0           485.0   
20638    -121.32     39.43                18.0       1860.0           409.0   
20639    -121.24     39.37                16.0       2785.0           616.0   

       population  households  median_income  median_house_valu

In [ ]:
print (data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB
None


* (2) 데이터 전처리

In [ ]:
# 범주형 변수 제외 :ocran_promimity 데이터분석에 필요하지 않음
data_clean = data.drop(columns=['ocean_proximity'])

# 결측치 처리 : 데이터에 극단적인 값이 있어도 크게 영향을 받지 않는 중앙값으로 설정
data_clean = data_clean.fillna(data_clean.median())

# 독립 변수(X):주택가격에 영향을 미칠 수 있는 변수
# 종속 변수(y):예측하고자 하는 주택 가격
X = data_clean.drop(columns=['median_house_value'])
y = data_clean['median_house_value']

# 데이터셋을 학습용과 테스트용으로 분할
# random_state =42: 코드를 여러 번 실행해도 항상 동일한 방식으로 데이터가 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

3. 모델 생성 및 학습

In [ ]:
# 상수항 추가 (statsmodels를 사용하기 위해 필요)
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# 선형 회귀 모델 생성 및 학습 (통계적 검증 포함)
lin_reg = sm.OLS(y_train, X_train_const).fit()

4. 가설 검정

In [ ]:
# 회귀 결과 요약 출력 (가설 검정 결과 포함)
print('선형 회귀 모델 (가설 검정 포함):')
print(lin_reg.summary())

선형 회귀 모델 (가설 검정 포함):
                            OLS Regression Results                            
Dep. Variable:     median_house_value   R-squared:                       0.640
Model:                            OLS   Adj. R-squared:                  0.640
Method:                 Least Squares   F-statistic:                     3669.
Date:                Wed, 30 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:15:27   Log-Likelihood:            -2.0749e+05
No. Observations:               16512   AIC:                         4.150e+05
Df Residuals:                   16503   BIC:                         4.151e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const          

###선형 회귀 모델 (가설 검정 포함)

① Dep. Variable(종속 변수): 회귀분석에서 예측하려는 변수 이름

② Model: 최소제곱법(OLS: Ordinary Least Squares)을 사용한 선형 회귀 모델

③ R-squared(결정 계수): 모델이 데이터를 얼마나 잘 설명하는지를 나타냄.
0에서 1 사이의 값을 가지며, 0.64라는 것은 모델이 종속 변수의 변동성 중 약 64%를 설명한다는 의미

④ Adj. R-squared(수정된 결정 계수): 결정 계수와 유사하지만, 독립 변수의 개수를 고려하여 조정된 값

⑤ Prob(F-statistic): F-statistic에 대한 p-값으로, 0.000이라는 것은 유의 수준(0.05)보다 작으므로, 모델 전체가 통계적으로 의미가 있다고 판단

⑥ 회귀계수(Coefficients)
const(상수항): 모든 독립 변수가 0일 때 주택 가격의 기본값

longitude(경도): 경도가 1도 증가할 때, 주택 가격이 약 42,630달러 감소

latitude(위도): 위도가 1도 증가할 때, 주택 가격이 약 42,450달러 감소

housing_median_age(주택 나이(중앙값)): 주택 연령이 1년 증가할 때, 주택 가격이 약 1,183달러 증가

total_rooms(전체 방 개수): 방의 개수가 1개 증가할 때, 주택 가격이 약 8.19달러 감소

total_bedrooms(전체 침실 개수): 침실 개수가 1개 증가할 때, 주택 가격이 약 116.26달러 증가

population(인구수): 인구가 1명 증가할 때, 주택 가격이 약 38.49달러 감소

households(가구 수): 가구 수가 1개 증가할 때, 주택 가격이 약 46.34달러 증가

median_income(중간 소득): 중간 소득이 1 단위(약 만 달러) 증가할 때, 주택 가격이 약 40,540달러 증가

## `random_state` 설명

`random_state`는 주로 데이터 분할, 모델 초기화 등 무작위성이 개입되는 작업에서 사용 `random_state`에 특정 정수 값을 설정하면, 해당 무작위 작업이 항상 동일한 결과를 생성하도록 보장

### `train_test_split`에서의 `random_state`

`sklearn.model_selection.train_test_split` 함수에서 `random_state`를 사용하면, 데이터를 훈련 세트와 테스트 세트로 분할할 때 무작위성이 고정

*   **재현성 (Reproducibility)**: `random_state`를 설정하지 않으면, `train_test_split`를 실행할 때마다 데이터가 다르게 분할됨, 이는 코드 실행 시마다 모델의 훈련 결과와 평가 지표가 달라질 수 있음을 의미

 `random_state`를 특정 정수(예: `42`)로 설정하면, 코드를 여러 번 실행해도 항상 동일한 방식으로 데이터가 분할되므로, 실험 결과를 재현

*   **일관된 비교**: 다른 모델이나 하이퍼파라미터를 비교할 때, 모든 모델이 동일한 훈련/테스트 데이터 분할을 사용하도록 보장하여 공정한 비교 가능

예를 들어, `random_state = 42`는 단순히 42라는 숫자에 특별한 의미가 있는 것이 아니라, 개발자가 임의로 선택한 정수이며 이 숫자를 사용하면 해당 무작위 작업의 시작점이 항상 동일하게 고정

## 상수항 (`const`) 설명

상수항(constant, 또는 절편(intercept))은 다음과 같은 이유로 통계 회귀 모델에 포함

1.  **모델의 기본값**: 모든 독립 변수(예: `longitude`, `latitude`, `housing_median_age` 등)가 0일 때, 종속 변수(`median_house_value`)의 예상되는 평균값을 나타냅니다. 즉, 독립 변수들의 영향이 없을 때의 종속 변수 기본 수준을 의미합니다.

2.  **`statsmodels` 라이브러리의 역할**: `statsmodels`와 같은 통계 패키지에서는 회귀 모델을 구축할 때 이 상수항을 자동으로 포함시키지 않는 경우가 많습니다. 따라서 모델이 데이터를 더 정확하게 설명할 수 있도록 **`sm.add_constant(X_train)`**과 같이 `add_constant` 함수를 사용하여 독립 변수 데이터 `X_train`에 상수항 열을 명시적으로 추가해 주어야 합니다. 이 함수가 상수항 값을 설정하는 역할을 합니다.

3.  **수학적 의미**: 선형 회귀 방정식 $y = \beta_0 + \beta_1x_1 + \beta_2x_2 + ... + \epsilon$ 에서 $\beta_0$ 에 해당합니다. 이 $\beta_0$가 바로 상수항이며, 다른 독립 변수들이 0일 때의 $y$ 값을 나타냅니다.